# Stage 3: PPO (TRL) – classical RLHF on GPT-2

In [ ]:
import os
import torch
import random
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from trl.core import LengthSampler

# ---------------------------
# Paths
# ---------------------------
SFT_CHECKPOINT = "./sft_gpt2_dolly/final"
RM_CHECKPOINT  = "./rm_gpt2_hh/final"
PPO_OUTPUT_DIR = "./ppo_gpt2_final"

MAX_PROMPT_LEN = 128
MAX_NEW_TOKENS = 64
NUM_PROMPTS    = 6000
BATCH_SIZE     = 4
MINI_BATCH     = 4
GRAD_ACCUM     = 1
LR             = 1.4e-6
KL_COEF        = 0.02
PPO_EPOCHS     = 1
SEED           = 42
TOTAL_STEPS    = 300

os.makedirs(PPO_OUTPUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"SFT checkpoint: {SFT_CHECKPOINT}")
print(f"RM  checkpoint: {RM_CHECKPOINT}")

# ---------------------------
# 1. Tokenizer
# ---------------------------
tokenizer = AutoTokenizer.from_pretrained(SFT_CHECKPOINT, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# ---------------------------
# 2. Policy (SFT + value head)
# ---------------------------
print("Loading policy (SFT + value head)...")
base_model = AutoModelForCausalLM.from_pretrained(SFT_CHECKPOINT, local_files_only=True)
policy = AutoModelForCausalLMWithValueHead(base_model)
policy.to(device)
policy.is_peft_model = False          # ← CRITICAL FIX

if hasattr(policy.pretrained_model, "gradient_checkpointing_enable"):
    policy.pretrained_model.gradient_checkpointing_enable()

# ---------------------------
# 3. Reward model
# ---------------------------
print("Loading reward model...")
reward_model = AutoModelForSequenceClassification.from_pretrained(
    RM_CHECKPOINT,
    num_labels=1,
    problem_type="regression",
    local_files_only=True,
)
reward_model.to(device)
reward_model.eval()
for p in reward_model.parameters():
    p.requires_grad = False

# ---------------------------
# 4. Reference model
# ---------------------------
print("Loading reference model...")
ref_base = AutoModelForCausalLM.from_pretrained(SFT_CHECKPOINT, local_files_only=True)
ref_model = AutoModelForCausalLMWithValueHead(ref_base)
ref_model.to(device)
ref_model.is_peft_model = False
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

# ---------------------------
# 5. Prompts
# ---------------------------
print("Preparing prompts from Dolly...")
dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
prompts = []
for ex in dolly:
    instr = ex["instruction"].strip()
    ctx = ex.get("context", "").strip()
    if ctx:
        text = f"Instruction: {instr}\nContext: {ctx}\nResponse:"
    else:
        text = f"Instruction: {instr}\nResponse:"
    prompts.append(text)

random.shuffle(prompts)
prompts = prompts[:NUM_PROMPTS]
print(f"Using {len(prompts)} prompts")

def collate_prompts(batch_prompts):
    tokenized = tokenizer(
        batch_prompts,
        truncation=True,
        max_length=MAX_PROMPT_LEN,
        padding=True,
        return_tensors="pt",
    )
    return {k: v.to(device) for k, v in tokenized.items()}

# ---------------------------
# 6. PPO Config
# ---------------------------
ppo_config = PPOConfig(
    model_name="gpt2",
    learning_rate=LR,
    batch_size=BATCH_SIZE,
    mini_batch_size=MINI_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    ppo_epochs=PPO_EPOCHS,
    init_kl_coef=KL_COEF,
    target_kl=6.0,
    seed=SEED,
    log_with=None,
)

# ---------------------------
# 7. PPO Trainer
# ---------------------------
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy,
    ref_model=ref_model,
    tokenizer=tokenizer,
)

# ---------------------------
# 8. Reward function
# ---------------------------
def compute_rewards(prompts, responses):
    texts = [p + r for p, r in zip(prompts, responses)]
    inputs = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_PROMPT_LEN + MAX_NEW_TOKENS,
        padding=True,
        return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        logits = reward_model(**inputs).logits.squeeze(-1)
    return logits.cpu().tolist()

# ---------------------------
# 9. Training loop
# ---------------------------
print("Starting PPO training...")
print(f"Batch size: {BATCH_SIZE} | KL coef: {KL_COEF} | LR: {LR}")
print(f"Max new tokens: {MAX_NEW_TOKENS} | Total steps: {TOTAL_STEPS}")

generation_kwargs = {
    "min_length": -1,
    "top_k": 50,
    "top_p": 0.95,
    "do_sample": True,
    "temperature": 1.0,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": MAX_NEW_TOKENS,
}

output_length_sampler = LengthSampler(32, MAX_NEW_TOKENS)

for step in tqdm(range(TOTAL_STEPS), desc="PPO"):
    batch_prompts = random.sample(prompts, BATCH_SIZE)
    batch = collate_prompts(batch_prompts)

    query_tensors = [batch["input_ids"][i] for i in range(BATCH_SIZE)]

    response_tensors = ppo_trainer.generate(
        query_tensors,
        return_prompt=False,
        length_sampler=output_length_sampler,
        **generation_kwargs,
    )

    responses = [tokenizer.decode(r.squeeze(), skip_special_tokens=True) for r in response_tensors]

    rewards = compute_rewards(batch_prompts, responses)
    reward_tensors = [torch.tensor(r, device=device) for r in rewards]

    stats = ppo_trainer.step(query_tensors, response_tensors, reward_tensors)

    if step % 20 == 0 or step == TOTAL_STEPS - 1:
        mean_reward = np.mean(rewards)
        kl = stats.get("objective/kl", 0.0)
        print(f"Step {step:4d} | mean reward {mean_reward:.3f} | KL {kl:.4f}")

# ---------------------------
# 10. Save
# ---------------------------
final_ppo_path = os.path.join(PPO_OUTPUT_DIR, "final")
ppo_trainer.model.save_pretrained(final_ppo_path)
tokenizer.save_pretrained(final_ppo_path)
print(f"\nPPO checkpoint saved to: {final_ppo_path}")

# ---------------------------
# 11. Quick check
# ---------------------------
print("\n--- Quick PPO generation check ---")
test_prompt = "Instruction: Explain what a black hole is in simple terms.\nResponse:"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = policy.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(out[0], skip_special_tokens=True))